# Comparaison de modèles VERIS → ATT&CK - Version 2 (Optimisée)

Ce notebook entraîne plusieurs modèles de classification multi-label pour prédire les techniques MITRE ATT&CK associées à une capacité VERIS.

**Optimisations majeures de la V2 pour maximiser l'accuracy :**
1. **Extraction de features hybride (Word + Char-level)** : Capture à la fois le vocabulaire sémantique (`TfidfVectorizer` mot) et la structure des identifiants ou acronymes techniques (`TfidfVectorizer` caractère au sein des mots).
2. **Classifieurs robustes au déséquilibre extrême** : Remplacement des SVM linéaires bruts par un modèle calibré en probabilité (`CalibratedClassifierCV`) et ajout de `ComplementNB` (conçu spécifiquement pour les jeux de données déséquilibrés).
3. **Optimisation de l'évaluation temporelle** pour simuler fidèlement une généralisation vers la version 19.1.

In [1]:
import sqlite3
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MultiLabelBinarizer, normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "databases").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from paths import MAPPINGS_DB, MAPPING_SETS

TRAIN_SETS = MAPPING_SETS[:3]
TEST_SET = MAPPING_SETS[3]
TOP_K = 10
RANDOM_STATE = 42

print(f"Projet : {ROOT}")
print(f"Entraînement : {TRAIN_SETS}")
print(f"Test         : {TEST_SET}")

Projet : C:\Users\simon\Desktop\UQAC\Uqac semestre 2\SIEM
Entraînement : [('9.0', '1.3.5', 'veris-1.3.5_attack-9.0-enterprise'), ('12.1', '1.3.7', 'veris-1.3.7_attack-12.1-enterprise'), ('16.1', '1.4.0', 'veris-1.4.0_attack-16.1-enterprise')]
Test         : ('19.1', '1.4.1', 'veris-1.4.1_attack-19.1-enterprise')


In [2]:
import sqlite3
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MultiLabelBinarizer, normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "databases").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from paths import MAPPINGS_DB, MAPPING_SETS

TRAIN_SETS = MAPPING_SETS[:3]
TEST_SET = MAPPING_SETS[3]
TOP_K = 10
RANDOM_STATE = 42

print(f"Projet : {ROOT}")
print(f"Entraînement : {TRAIN_SETS}")
print(f"Test         : {TEST_SET}")

Projet : C:\Users\simon\Desktop\UQAC\Uqac semestre 2\SIEM
Entraînement : [('9.0', '1.3.5', 'veris-1.3.5_attack-9.0-enterprise'), ('12.1', '1.3.7', 'veris-1.3.7_attack-12.1-enterprise'), ('16.1', '1.4.0', 'veris-1.4.0_attack-16.1-enterprise')]
Test         : ('19.1', '1.4.1', 'veris-1.4.1_attack-19.1-enterprise')


## 1. Chargement et préparation des données

In [5]:
def load_mappings(db_path: Path, veris_version: str, attack_version: str) -> pd.DataFrame:
    query = """
        SELECT
            m.capability_id,
            m.capability_description,
            m.capability_group,
            m.attack_object_id,
            m.attack_object_name
        FROM mappings m
        JOIN mapping_sets ms ON m.mapping_set_id = ms.id
        WHERE ms.mapping_framework_version = ?
          AND ms.attack_version = ?
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn, params=(veris_version, attack_version))
    df["capability_key"] = df["capability_id"].str.lower()
    return df


def aggregate_by_capability(df: pd.DataFrame) -> pd.DataFrame:
    """Regroupe les techniques ATT&CK par capacité VERIS."""
    grouped = (
        df.groupby("capability_key", as_index=False)
        .agg(
            capability_id=("capability_id", "first"),
            capability_description=("capability_description", "first"),
            capability_group=("capability_group", "first"),
            attack_object_ids=("attack_object_id", lambda s: sorted(set(s))),
        )
    )
    grouped["text"] = (
        grouped["capability_group"].fillna("")
        + " "
        + grouped["capability_id"].fillna("")
        + " "
        + grouped["capability_description"].fillna("")
    ).str.strip()
    return grouped


train_frames = [
    load_mappings(MAPPINGS_DB, veris, attack)
    for attack, veris, _ in TRAIN_SETS
]
train_raw = pd.concat(train_frames, ignore_index=True)

test_attack, test_veris, _ = TEST_SET
test_raw = load_mappings(MAPPINGS_DB, test_veris, test_attack)

train_df = aggregate_by_capability(train_raw)
test_df = aggregate_by_capability(test_raw)

# Espace d'étiquettes = techniques ATT&CK présentes dans le jeu de test (19.1)
label_space = sorted(test_df["attack_object_ids"].explode().dropna().unique())


def filter_labels(techniques: list[str]) -> list[str]:
    return [t for t in techniques if t in label_space]


train_df["labels"] = train_df["attack_object_ids"].apply(filter_labels)
test_df["labels"] = test_df["attack_object_ids"].apply(filter_labels)

# Retirer les exemples sans étiquette dans l'espace cible
train_df = train_df[train_df["labels"].map(len) > 0].reset_index(drop=True)
test_df = test_df[test_df["labels"].map(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer(classes=label_space)
y_train = mlb.fit_transform(train_df["labels"])
y_test = mlb.transform(test_df["labels"])
X_train_text = train_df["text"].tolist()
X_test_text = test_df["text"].tolist()

overlap = set(train_df["capability_key"]) & set(test_df["capability_key"])

print(f"Exemples d'entraînement : {len(train_df)}")
print(f"Exemples de test        : {len(test_df)}")
print(f"Espace d'étiquettes (ATT&CK 19.1) : {len(label_space)} techniques")

Exemples d'entraînement : 157
Exemples de test        : 176
Espace d'étiquettes (ATT&CK 19.1) : 589 techniques


## 2. Vectorisation avancée (Hybride Word + Char Word-Bound)

In [6]:
# Extraction au niveau des mots
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

# Extraction au niveau des caractères (idéal pour les sous-chaînes de codes ou d'IDs techniques)
char_vectorizer = TfidfVectorizer(
    ngram_range=(3, 5),
    analyzer="char_wb",
    min_df=2,
    sublinear_tf=True
)

# Fusion des deux espaces de features
vectorizer = FeatureUnion([
    ("word", word_vectorizer),
    ("char", char_vectorizer)
])

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Matrice TF-IDF combinée V2 : {X_train.shape} -> {X_test.shape}")

Matrice TF-IDF combinée V2 : (157, 4752) -> (176, 4752)


## 3. Fonctions d'évaluation

In [7]:
def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int = TOP_K) -> float:
    precisions = []
    for i in range(y_true.shape[0]):
        true_idx = set(np.where(y_true[i] == 1)[0])
        if not true_idx:
            continue
        top_k = np.argsort(scores[i])[::-1][:k]
        pred_idx = set(top_k)
        precisions.append(len(true_idx & pred_idx) / k)
    return float(np.mean(precisions)) if precisions else 0.0


def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int = TOP_K) -> float:
    recalls = []
    for i in range(y_true.shape[0]):
        true_idx = set(np.where(y_true[i] == 1)[0])
        if not true_idx:
            continue
        top_k = np.argsort(scores[i])[::-1][:k]
        pred_idx = set(top_k)
        recalls.append(len(true_idx & pred_idx) / len(true_idx))
    return float(np.mean(recalls)) if recalls else 0.0


def evaluate_model(name: str, y_pred: np.ndarray, y_scores: np.ndarray, train_s: float, infer_s: float) -> dict:
    return {
        "modèle": name,
        "subset_accuracy": accuracy_score(y_test, y_pred),
        "f1_micro": f1_score(y_test, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_samples": f1_score(y_test, y_pred, average="samples", zero_division=0),
        "jaccard_samples": jaccard_score(y_test, y_pred, average="samples", zero_division=0),
        "hamming_loss": hamming_loss(y_test, y_pred),
        f"precision@{TOP_K}": precision_at_k(y_test, y_scores, TOP_K),
        f"recall@{TOP_K}": recall_at_k(y_test, y_scores, TOP_K),
        "train_time_s": train_s,
        "infer_time_s": infer_s,
        "infer_ms_per_sample": (infer_s / len(y_test)) * 1000,
    }


def scores_from_estimator(model, X) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


results: list[dict] = []

## 4. Entraînement des modèles optimisés V2

In [14]:
def fit_and_evaluate(name: str, estimator) -> dict:
    print(f"Entraînement : {name}...")
    t0 = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = estimator.predict(X_test)
    y_scores = scores_from_estimator(estimator, X_test)
    infer_s = time.perf_counter() - t1

    metrics = evaluate_model(name, y_pred, y_scores, train_s, infer_s)
    results.append(metrics)
    print(
        f"  F1 micro={metrics['f1_micro']:.3f} | "
        f"Jaccard={metrics['jaccard_samples']:.3f} | "
        f"P@{TOP_K}={metrics[f'precision@{TOP_K}']:.3f} | "
        f"train={train_s:.1f}s"
    )
    return metrics


# --- Modèle 1 : Régression logistique (OvR - Balanced) ---
fit_and_evaluate(
    "Logistic Regression (OvR - Balanced)",
    OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        n_jobs=-1,
    ),
)

# --- Modèle 2 : SVM Linéaire (OvR - Balanced) ---
# Version corrigée pour éliminer l'erreur Multi-Label de CalibratedClassifierCV
fit_and_evaluate(
    "Linear SVM (OvR - Balanced)",
    OneVsRestClassifier(
        LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
        n_jobs=-1,
    ),
)

# --- Modèle 3 : Complement Naive Bayes (Spécifique aux classes fortement déséquilibrées) ---
fit_and_evaluate(
    "Complement Naive Bayes (OvR)",
    OneVsRestClassifier(ComplementNB(), n_jobs=-1),
)

Entraînement : Logistic Regression (OvR - Balanced)...
  F1 micro=0.693 | Jaccard=0.592 | P@10=0.353 | train=1.0s
Entraînement : Linear SVM (OvR - Balanced)...
  F1 micro=0.756 | Jaccard=0.666 | P@10=0.345 | train=0.4s
Entraînement : Complement Naive Bayes (OvR)...
  F1 micro=0.339 | Jaccard=0.226 | P@10=0.343 | train=0.6s


{'modèle': 'Complement Naive Bayes (OvR)',
 'subset_accuracy': 0.11931818181818182,
 'f1_micro': 0.33904761904761904,
 'f1_macro': 0.1569039486097311,
 'f1_samples': 0.2586678292223721,
 'jaccard_samples': 0.22570357522764342,
 'hamming_loss': 0.010042058959716006,
 'precision@10': 0.34261363636363634,
 'recall@10': 0.7113638368521336,
 'train_time_s': 0.5532293999999638,
 'infer_time_s': 0.37549380000018573,
 'infer_ms_per_sample': 2.1334875000010554}

## 5. Comparaison et Résultats V2

In [10]:
results_df = pd.DataFrame(results).sort_values("f1_micro", ascending=False)
display_cols = [
    "modèle",
    "f1_micro",
    "f1_macro",
    "jaccard_samples",
    f"precision@{TOP_K}",
    f"recall@{TOP_K}",
    "hamming_loss",
    "train_time_s",
]
results_df[display_cols].round(4)

,modèle,f1_micro,f1_macro,jaccard_samples,precision@10,recall@10,hamming_loss,train_time_s
0,Logistic Regression (OvR - Balanced),0.693,0.7129,0.592,0.3528,0.731,0.0097,8.7262
1,Logistic Regression (OvR - Balanced),0.693,0.7129,0.592,0.3528,0.731,0.0097,6.8627
